# ui

> Everything that hacks the JupyterLab frontend via [ipylab](https://github.com/jtpio/ipylab): the side panel, the notebook toolbar buttons and helpers to insert cells. This is the only module that talks to the frontend, so any future migration (e.g. to the [fleming79 ipylab fork](https://github.com/fleming79/ipylab) for per-cell buttons) stays contained here.

In [ ]:
#| default_exp ui

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import atexit
from ipylab import Panel
from ipywidgets import Button, Dropdown, Label, Text, FloatSlider, IntSlider

YASI_BUTTONS = ("yasi-u", "yasi-a", "yasi-send")

class YasiUI:
    """Builds yasi's JupyterLab UI: a side panel with model/parameter widgets and notebook toolbar buttons.
    Talks back to its owner through callbacks, so it contains no chat logic itself."""
    def __init__(self,
                 app, # ipylab.JupyterFrontEnd instance
                 models: list, # list of model dicts with keys company/name/id
                 on_send_dialog, # callback, called when the send toolbar button is clicked
                 on_model_change=None, # callback(model_id), called when a model is selected
                 on_params_change=None, # callback(max_tokens, temperature, presence_penalty)
                 tag_user : str='#| chat_user', # tag for user chat markdown cells
                 tag_assistant : str='#| chat_assistant' # tag for assistant chat markdown cells
                ):
        self.app = app
        self.models = models
        self.on_send_dialog = on_send_dialog
        self.on_model_change = on_model_change
        self.on_params_change = on_params_change
        self.tag_user = tag_user
        self.tag_assistant = tag_assistant

        self.remove_stale_toolbar_buttons()
        self.setup_ipylab_panel()
        self.setup_ipylab_toolbar()
        # remove the UI when the kernel shuts down or restarts cleanly
        atexit.register(self.cleanup)

    def setup_ipylab_panel(self):
        """Add an ipylab panel to browse and select available models"""
        companies = tuple(sorted({m['company'] for m in self.models}, key=str.casefold))
        company = 'Meta' if 'Meta' in companies else companies[0]
        filtered_models = tuple(sorted([m['id'] for m in self.models if m['company'] == company], key=str.casefold))
        my_fav_model = 'meta-llama/llama-3.1-8b-instruct'
        model = my_fav_model if my_fav_model in filtered_models else filtered_models[0]
        self.wgt_dd_company = Dropdown(options=companies, value=company, description='Company', tooltip='', layout={'width': '600px'})
        self.wgt_dd_company.observe(self.update_any_widget)
        self.wgt_dd_model = Dropdown(options=filtered_models, value=model, description='Model', tooltip='', layout={'width': '600px'})
        self.wgt_dd_model.observe(self.update_model_widget)
        self.wgt_txt_search_models = Text(value="", description='Search',disabled=False,indent=False)
        self.wgt_txt_search_models.observe(self.update_any_widget)
        self.wgt_btn_close_panel =  Button(button_style='danger',tooltip='Close Panel',icon='window-close ')
        self.wgt_btn_close_panel.on_click(self.remove_ipylab_features)

        # Openai Body parameters
        self.wgt_is_max_tokens = IntSlider(value=256, min=16, max=4096, description="Max tokens:", step=1)
        self.wgt_is_max_tokens.observe(self.update_any_widget)
        self.wgt_fs_temperature = FloatSlider(value=0.2, min=0.0, max=2.0, description="Temperature:", step=0.1, continuous_update=False)
        self.wgt_fs_temperature.observe(self.update_any_widget)
        self.wgt_fs_presence_penalty = FloatSlider(value=0.0, min=-2.0, max=2.0, description="Presence penalty:", step=0.1, continuous_update=False)
        self.wgt_fs_presence_penalty.observe(self.update_any_widget)

        # token/cost display, updated after every exchange
        self.wgt_lbl_usage = Label(value='')

        panel = Panel()
        panel.children = [self.wgt_btn_close_panel, self.wgt_dd_company, self.wgt_dd_model, #self.wgt_txt_search_models,
                          self.wgt_is_max_tokens, self.wgt_fs_temperature, self.wgt_fs_presence_penalty,
                          self.wgt_lbl_usage]
        panel.title.label = 'Yasi'
        panel.title.icon_class = 'fa fa-robot'
        panel.title.closable = True
        self.panel = panel
        self.app.shell.add(self.panel, 'right', { 'rank': '1000'})

    def add_message_cell(self, id):
        content =f"{self.tag_assistant}\n\n" if id == "yasi-a" else f"{self.tag_user}\n\n"
        self.create_new_markdown_cell(content)

    def setup_ipylab_toolbar(self):
        self.app.commands.add_command("add_message_cell", self.add_message_cell)
        self.app.commands.add_command("send_yasi_dialog", self.on_send_dialog)

        self.app.toolbar.add_button("yasi-u", "add_message_cell", { "id" : "yasi-u" }, iconClass='fa fa-comment', tooltip="Yasi: Add user message", after = "cellType")
        self.app.toolbar.add_button("yasi-a", "add_message_cell", { "id" : "yasi-a" }, iconClass='fa fa-robot', tooltip="Yasi: Add assistant message", after = "yasi-u")
        self.app.toolbar.add_button("yasi-send", "send_yasi_dialog", { "id" : "yasi-send" }, iconClass='fa fa-paper-plane', tooltip="Yasi: Send dialog to AI", after = "yasi-a")

    def show_usage(self, text:str):
        """Updates the usage line in the panel"""
        self.wgt_lbl_usage.value = text

    def remove_stale_toolbar_buttons(self):
        """Removes yasi toolbar buttons left over from a previous kernel.
        The frontend logs (and otherwise ignores) unknown names, so this is safe on a fresh start."""
        for name in YASI_BUTTONS:
            self.app.toolbar.send({"func": "removeToolbarButton", "payload": {"name": name}})

    def cleanup(self, change=None):
        """Removes all yasi UI elements, tolerating already-removed ones"""
        try:
            self.panel.close()
        except Exception:
            pass
        self.remove_stale_toolbar_buttons()

    def remove_ipylab_features(self, change):
        self.cleanup(change)
        atexit.unregister(self.cleanup)

    def update_any_widget(self, change):
        """Update the options of the widgets based on the selection"""
        selected_company = self.wgt_dd_company.value
        search = self.wgt_txt_search_models.value
        model_ids = tuple(sorted([m['id'] for m in self.models if (m['company'] == selected_company) & (search in m['id'])],
                                 key=str.casefold))
        self.wgt_dd_model.options = model_ids

        if self.on_params_change is not None:
            self.on_params_change(self.wgt_is_max_tokens.value,
                                  self.wgt_fs_temperature.value,
                                  self.wgt_fs_presence_penalty.value)

    def update_model_widget(self, change):
        if self.on_model_change is not None:
            self.on_model_change(self.wgt_dd_model.value)

    def create_new_markdown_cell(self,
                                 content: str # Markdown content
                                ):
        """Adds a new markdown cell with the given content below"""
        self.app.commands.execute('notebook:insert-cell-below')
        self.app.commands.execute('notebook:replace-selection', { 'text': content})
        self.app.commands.execute('notebook:change-cell-to-markdown')

The UI needs a running JupyterLab frontend, so it can only be exercised interactively:

In [ ]:
#| notest
from ipylab import JupyterFrontEnd
app = JupyterFrontEnd()
ui = YasiUI(app, models=[{'company': 'Meta', 'name': 'llama-3.1-8b-instruct', 'id': 'meta-llama/llama-3.1-8b-instruct'}],
            on_send_dialog=lambda id=None: print('send!'))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()